In [ ]:
!pip install transformers datasets

In [ ]:
import re
import random
from datasets import load_dataset, Dataset, DatasetDict
from transformers import T5Tokenizer
import pandas as pd
from typing import List, Dict, Optional
import nltk
from nltk.tokenize import sent_tokenize
import requests
import json
import numpy as np
import torch
from datasets import load_from_disk
from transformers import (
    T5Tokenizer,
    T5Config,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
import torch.nn as nn
from transformers.modeling_outputs import BaseModelOutput

In [ ]:
def create_unpunctuated_text(text: str) -> str:
    """Remove punctuation and capitalization from text"""
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def add_extra_spaces(text: str) -> str:
    """Add extra spaces randomly"""
    words = text.split()
    result = []
    for word in words:
        result.append(word)
        if random.random() < 0.3:  # 30% chance to add extra space
            result.append(' ')
    return ' '.join(result)

def remove_some_spaces( text: str) -> str:
    """Remove some spaces randomly"""
    words = text.split()
    result = []
    for i, word in enumerate(words):
        result.append(word)
        if i < len(words) - 1 and random.random() < 0.2:  # 20% chance to remove space
            continue
        else:
            result.append(' ')
    return ''.join(result).strip()

def mix_case_randomly(text: str) -> str:
    """Randomly mix uppercase and lowercase"""
    return ''.join(c.upper() if random.random() < 0.1 else c.lower() for c in text)


def load_wikipedia_dataset(language: str = "en", max_samples: Optional[int] = 10000):
    nltk.download('punkt')
    nltk.download('punkt_tab')
    try:
        wiki_dataset = load_dataset("wikimedia/wikipedia", f"20231101.{language}", split="train", streaming=True)

        samples = []
        count = 0

        for example in wiki_dataset:
            if max_samples and count >= max_samples:
                break

            text = example['text']
            sentences = sent_tokenize(text)

            for sentence in sentences:
                if len(sentence.split()) > 5:
                    # Apply transformations sequentially
                    input_text = create_unpunctuated_text(sentence)
                    input_text = add_extra_spaces(input_text)
                    input_text = remove_some_spaces(input_text)
                    input_text = mix_case_randomly(input_text)

                    if input_text.strip() and input_text != sentence:
                        samples.append({
                            "input_text": f"normalize: {input_text}",
                            "target_text": sentence
                        })
                        count += 1

            if count % 1000 == 0:
                print(f"Processed {count} samples...")

        dataset = Dataset.from_list(samples)
        print(f" Created Wikipedia dataset with {len(dataset)} samples")
        return dataset

    except Exception as e:
        print(f"Error loading Wikipedia dataset: {e}")
        return None


wiki_dataset = load_wikipedia_dataset(max_samples=100000)
train_test_split = wiki_dataset.train_test_split(test_size=0.2, seed=42)

print(f"\nFinal dataset statistics:")
print(f"Training samples: {len(train_test_split['train'])}")
print(f"Validation samples: {len(train_test_split['test'])}")

# Save datasets
train_test_split['train'].save_to_disk("./large_text_normalization_train")
train_test_split['test'].save_to_disk("./large_text_normalization_val")

# Show sample
print("\nSample from dataset:")
for i in range(3):
    sample = train_test_split['train'][i]
    print(f"Input:  {sample['input_text']}")
    print(f"Output: {sample['target_text']}")
    print()


In [ ]:

train_dataset = load_from_disk("/content/large_text_normalization_train")
test_dataset  = load_from_disk("/content/large_text_normalization_val")


model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)


def preprocess(examples):
    inputs = examples["input_text"]
    targets = examples["target_text"]

    model_inputs = tokenizer(inputs,
                             truncation=True, padding="max_length",max_length=256)

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets,
                           truncation=True, padding="max_length",max_length=256)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train_dataset.map(preprocess, batched=True)
test_dataset  = test_dataset.map(preprocess, batched=True)


data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)


def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Decode
    preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    refs = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Exact string match accuracy
    correct = sum(p.strip() == r.strip() for p, r in zip(preds, refs))
    acc = correct / len(preds)

    return {"accuracy": acc}


training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-text-normalization",
    #evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_dir="./logs",
    logging_steps=100,
    report_to="none"
)


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

print("Evaluating on test set...")
results = trainer.evaluate()
print(results)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import textwrap

# Load model
model_path = "/content/t5-text-normalization/checkpoint-20008/"   # adjust to your path
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# Input transcript
raw_transcript = """
As the morning breeze drifts through the trees, the sound of birds fills the air and people slowly begin their day, while the city awakens with the rhythm of footsteps, voices, and the gentle hum of life continuing in every direction."""

# Prefix "normalize:" if your training used that
input_text = "normalize: " + raw_transcript.strip()

# Tokenize input
inputs = tokenizer(input_text, return_tensors="pt", truncation=True)

# Generate output
with torch.no_grad():
    generated_ids = model.generate(
    **inputs,
    max_length=1024,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.9
)

chunks = textwrap.wrap(raw_transcript, 400)
normalized_chunks = []
for ch in chunks:
    input_text = "normalize: " + ch
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True)
    output_ids = model.generate(**inputs, max_length=256, num_beams=5)
    normalized_chunks.append(tokenizer.decode(output_ids[0], skip_special_tokens=True))

final_output = " ".join(normalized_chunks)

print("===== RAW TRANSCRIPT =====")
print(raw_transcript)
print("\n===== NORMALIZED OUTPUT =====")
print(final_output)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
    # Assuming 'my_output.txt' is a file you've created in Colab
    # You can replace this with the actual path to your file
    source_file_path = "/content/t5-text-normalization/checkpoint-20008/"

    # Define the destination path in your Google Drive
    destination_path = "/content/drive/MyDrive/t5-text-normalization/checkpoint-20008/"

    # Copy the file
    !cp -r "{source_file_path}" "{destination_path}"